In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'beverage_sales'
SOURCE_SCHEMA_NAME = 'silver'
SOURCE_TABLE_NAME = 'sales'

TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'dim_package'

UNKNOWN_KEY = -1

In [0]:
df_unknown_member = spark.createDataFrame(
    [(UNKNOWN_KEY, 'UNKNOWN', 'UNKNOWN', 'NONE', 'UNK', 'UNKNOWN')],
    'package_key bigint, package_name string, package_name_std string, '
    'package_marker string, package_category string, package_category_desc string'
)

In [0]:
df_source = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}')

## Cardinality check

`dropDuplicates` on a natural key is only safe when that key functionally
determines the remaining attributes. A future file where one `package_name` carries two different
attribute sets fails here instead of silently keeping an arbitrary row.

In [0]:
df_invalid_mapping = (
    df_source
    .groupBy('package_name')
    .agg(
        F.countDistinct(
            F.struct(
                'package_name_std',
                'package_marker',
                'package_category',
                'package_category_desc'
            )
        ).alias('attribute_count')
    )
    .filter(F.col('attribute_count') > 1)
)

assert df_invalid_mapping.count() == 0, 'package_name does not uniquely determine its attributes'

In [0]:
df_dim_package = (
    df_source
    .select(
        'package_name',
        'package_name_std',
        'package_marker',
        'package_category',
        'package_category_desc'
    )
    .dropDuplicates(['package_name'])
    .withColumn('package_key', F.abs(F.xxhash64(F.col('package_name'))))
    .select(
        'package_key',
        'package_name',
        'package_name_std',
        'package_marker',
        'package_category',
        'package_category_desc'
    )
    .unionByName(df_unknown_member)
)

In [0]:
df_dim_package\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')